In [26]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

model_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

# Hold out entire participants, not individual rows
participant_ids = sorted(evaluation_data["Id"].unique())

test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

test_data = evaluation_data[
    evaluation_data["Id"].isin(test_ids)
].copy()

print("Train participants:", train_data["Id"].nunique())
print("Test participants:", test_data["Id"].nunique())
print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Test participant IDs:", test_ids)

Train participants: 12
Test participants: 3
Train rows: 65
Test rows: 9
Test participant IDs: [np.int64(7086361926), np.int64(8378563200), np.int64(8792009665)]


In [27]:
X_train = train_data[model_features]
X_test = test_data[model_features]

model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

model.fit(X_train)

test_data["anomaly_label"] = model.predict(X_test)
test_data["anomaly_score"] = model.decision_function(X_test)

print("Test observations:", len(test_data))
print("\nPredictions:")
print(test_data["anomaly_label"].value_counts())

display(
    test_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "anomaly_label",
            "anomaly_score"
        ]
    ].sort_values("anomaly_score")
)

Test observations: 9

Predictions:
anomaly_label
-1    6
 1    3
Name: count, dtype: int64


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,anomaly_label,anomaly_score
176,7086361926,2016-04-10,-93.127545,-87.755102,-1,-0.214813
178,7086361926,2016-04-12,-94.806405,-3.854875,-1,-0.100385
190,8378563200,2016-04-12,-68.580871,-49.413690,-1,-0.095372
188,8378563200,2016-04-10,-74.533663,19.498227,-1,-0.056152
198,8792009665,2016-04-09,-68.707900,29.037520,-1,-0.051541
187,8378563200,2016-04-09,-78.318284,9.571857,-1,-0.037800
186,8378563200,2016-04-08,-19.177533,-34.333242,1,0.032205
189,8378563200,2016-04-11,-14.463696,-26.506681,1,0.067199
177,7086361926,2016-04-11,-24.608324,16.326531,1,0.079745


In [28]:
test_data["rule_alert"] = (
    (test_data["steps_deviation_7d_pct"] <= -10)
    | (test_data["sleep_deviation_7d_pct"] <= -10)
)

ml_flag = test_data["anomaly_label"] == -1
rule_flag = test_data["rule_alert"]

print("ML anomalies:", ml_flag.sum())
print("Rule alerts:", rule_flag.sum())
print("Both flag:", (ml_flag & rule_flag).sum())
print("ML only:", (ml_flag & ~rule_flag).sum())
print("Rule only:", (~ml_flag & rule_flag).sum())

ML anomalies: 6
Rule alerts: 9
Both flag: 6
ML only: 0
Rule only: 3


## Day 11 — Participant-Level Model Evaluation

### Evaluation Setup
- Training set: 12 participants, 65 observations.
- Test set: 3 unseen participants, 9 observations.
- Isolation Forest was fitted exclusively on the training participants.
- Features: percentage deviation in daily steps and recorded sleep from personal baselines.

### Results
- Isolation Forest flagged 6 of 9 test observations as anomalies.
- The exploratory rule-based method flagged 9 of 9 observations.
- Both methods flagged 6 observations; 3 were flagged only by the rule.
- No observations were flagged exclusively by Isolation Forest.

### Limitations
- The test set is small and contains unequal participant observation counts.
- Ground-truth anomaly or clinical outcome labels are unavailable.
- Recorded sleep may reflect incomplete tracking rather than actual sleep duration.
- These results measure agreement between exploratory methods, not diagnostic accuracy or clinical validity.

### Conclusion
Participant-level holdout evaluation provides an initial check of anomaly detection on unseen participants. Larger datasets, stronger data-quality checks, repeated participant-level validation and independent outcome labels are needed before making performance or health-related claims.

In [29]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

candidate_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct",
    "steps_rolling_3d",
    "sleep_rolling_3d",
    "Calories",
    "VeryActiveMinutes",
    "SedentaryMinutes"
]

X = evaluation_data[candidate_features]

print("Shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

print("\nFeature statistics:")
display(X.describe().round(2))

print("\nFeature correlation:")
display(X.corr().round(2))

Shape: (74, 7)

Missing values:
steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
steps_rolling_3d          0
sleep_rolling_3d          0
Calories                  0
VeryActiveMinutes         0
SedentaryMinutes          0
dtype: int64

Feature statistics:


,steps_deviation_7d_pct,sleep_deviation_7d_pct,steps_rolling_3d,sleep_rolling_3d,Calories,VeryActiveMinutes,SedentaryMinutes
count,74.00,74.00,74.00,74.00,74.00,74.00,74.00
mean,-16.32,-4.11,8023.39,469.00,2209.72,17.70,688.50
std,37.88,31.78,3445.64,98.86,755.93,23.91,228.85
min,-99.22,-95.55,1112.67,180.33,600.00,0.00,61.00
25%,-37.77,-19.82,5501.75,401.25,1802.00,0.00,642.00
50%,-9.30,-2.00,7412.17,469.00,2188.00,7.50,731.50
75%,6.85,14.08,11099.50,532.33,2619.50,30.75,801.00
max,67.63,68.45,14480.67,685.00,3817.00,123.00,1368.00



Feature correlation:


,steps_deviation_7d_pct,sleep_deviation_7d_pct,steps_rolling_3d,sleep_rolling_3d,Calories,VeryActiveMinutes,SedentaryMinutes
steps_deviation_7d_pct,1.00,-0.08,0.43,0.05,0.45,0.14,0.27
sleep_deviation_7d_pct,-0.08,1.00,-0.10,0.35,-0.03,-0.05,-0.28
steps_rolling_3d,0.43,-0.10,1.00,-0.05,0.07,0.30,0.10
sleep_rolling_3d,0.05,0.35,-0.05,1.00,-0.18,-0.01,-0.27
Calories,0.45,-0.03,0.07,-0.18,1.00,0.28,0.47
VeryActiveMinutes,0.14,-0.05,0.30,-0.01,0.28,1.00,-0.09
SedentaryMinutes,0.27,-0.28,0.10,-0.27,0.47,-0.09,1.00


In [30]:
evaluation_data = evaluation_data.sort_values(
    ["Id", "date"]
).copy()

evaluation_data["steps_previous_3obs"] = (
    evaluation_data.groupby("Id")["TotalSteps"]
    .transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
)

evaluation_data["sleep_previous_3obs"] = (
    evaluation_data.groupby("Id")["recorded_sleep_minutes"]
    .transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
)

display(
    evaluation_data[
        [
            "Id",
            "date",
            "TotalSteps",
            "steps_previous_3obs",
            "recorded_sleep_minutes",
            "sleep_previous_3obs"
        ]
    ].head(10)
)

,Id,date,TotalSteps,steps_previous_3obs,recorded_sleep_minutes,sleep_previous_3obs
7,1503960366,2016-04-02,11248,NaN,591,NaN
8,1503960366,2016-04-03,10016,11248.000000,480,591.000000
9,1503960366,2016-04-04,14557,10632.000000,392,535.500000
10,1503960366,2016-04-05,14844,11940.333333,343,487.666667
11,1503960366,2016-04-09,12432,13139.000000,437,405.000000
12,1503960366,2016-04-10,10057,13944.333333,503,390.666667
13,1503960366,2016-04-11,10990,12444.333333,351,427.666667
25,1927972279,2016-04-10,2105,NaN,516,NaN
26,1927972279,2016-04-11,1209,2105.000000,559,516.000000
27,1927972279,2016-04-12,24,1657.000000,514,537.500000


In [31]:
print("Missing previous-step averages:",
      evaluation_data["steps_previous_3obs"].isna().sum())

print("Missing previous-sleep averages:",
      evaluation_data["sleep_previous_3obs"].isna().sum())

print("\nRows without previous observations:")
display(
    evaluation_data.loc[
        evaluation_data["steps_previous_3obs"].isna()
        | evaluation_data["sleep_previous_3obs"].isna(),
        ["Id", "date", "TotalSteps", "recorded_sleep_minutes"]
    ]
)

Missing previous-step averages: 15
Missing previous-sleep averages: 15

Rows without previous observations:


,Id,date,TotalSteps,recorded_sleep_minutes
7,1503960366,2016-04-02,11248,591
25,1927972279,2016-04-10,2105,516
35,2026352035,2016-04-08,3762,579
47,2347167796,2016-04-05,10114,510
61,3977333714,2016-04-08,10226,657
72,4020332650,2016-03-31,7428,281
87,4319703577,2016-04-05,8359,502
97,4445114986,2016-04-08,5543,251
111,4702921684,2016-04-05,6670,479
125,5553957443,2016-04-08,9938,458


In [32]:
from sklearn.ensemble import IsolationForest

improved_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct",
    "steps_previous_3obs",
    "sleep_previous_3obs"
]

model_data = evaluation_data.dropna(
    subset=improved_features
).copy()

X_improved = model_data[improved_features]

print("Original observations:", len(evaluation_data))
print("Usable observations:", len(model_data))
print("Excluded observations:", len(evaluation_data) - len(model_data))
print("Missing values:\n", X_improved.isna().sum())

Original observations: 74
Usable observations: 59
Excluded observations: 15
Missing values:
 steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
steps_previous_3obs       0
sleep_previous_3obs       0
dtype: int64


In [33]:
improved_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

improved_model.fit(X_improved)

model_data["improved_label"] = improved_model.predict(X_improved)
model_data["improved_score"] = improved_model.decision_function(X_improved)

print("\nImproved model predictions:")
print(model_data["improved_label"].value_counts())

display(
    model_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "improved_label",
            "improved_score"
        ]
    ].sort_values("improved_score").head(10)
)


Improved model predictions:
improved_label
 1    45
-1    14
Name: count, dtype: int64


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,improved_label,improved_score
177,7086361926,2016-04-11,-24.608324,16.326531,-1,-0.173198
51,2347167796,2016-04-09,62.925707,-95.549339,-1,-0.129031
27,1927972279,2016-04-12,-99.220237,54.885923,-1,-0.105033
52,2347167796,2016-04-10,2.105918,29.425229,-1,-0.075854
26,1927972279,2016-04-11,-60.719424,68.445975,-1,-0.069160
178,7086361926,2016-04-12,-94.806405,-3.854875,-1,-0.055266
115,4702921684,2016-04-09,67.628397,57.893258,-1,-0.029842
12,1503960366,2016-04-10,-22.756449,32.468021,-1,-0.020374
129,5553957443,2016-04-12,-62.873549,-34.804754,-1,-0.015776
78,4020332650,2016-04-10,-19.357157,-83.338694,-1,-0.011821


## Day 12 — Feature Engineering Experiment

### Objective
Investigate whether previous-observation activity and sleep trends can complement personal-baseline deviation features in anomaly detection.

### Features
- Steps deviation from the 7-observation personal baseline
- Recorded sleep deviation from the 7-observation personal baseline
- Previous 3 available observations' average steps
- Previous 3 available observations' average recorded sleep

### Results
- Original evaluation observations: 74
- Observations excluded due to unavailable previous-observation averages: 15
- Usable observations: 59
- Isolation Forest flagged 14 observations as anomalies and 45 as normal.

### Limitations
- The model was fitted and inspected on the same 59 observations; this is not independent evaluation.
- No ground-truth anomaly labels are available.
- Recorded sleep may be affected by incomplete wearable tracking.
- Results cannot establish clinical significance or prove that the expanded feature set is better.

### Conclusion
Previous-observation features provide an additional representation of recent activity and sleep patterns. A fair comparison requires the same participant-level train/test split and a consistent evaluation protocol for both feature sets.

In [34]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

evaluation_data = evaluation_data.sort_values(
    ["Id", "date"]
).copy()

evaluation_data["steps_previous_3obs"] = (
    evaluation_data.groupby("Id")["TotalSteps"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

evaluation_data["sleep_previous_3obs"] = (
    evaluation_data.groupby("Id")["recorded_sleep_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# Same participant-level split used on Day 11
participant_ids = sorted(evaluation_data["Id"].unique())
test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

test_data = evaluation_data[
    evaluation_data["Id"].isin(test_ids)
].copy()

print("Training participants:", train_data["Id"].nunique())
print("Test participants:", test_data["Id"].nunique())
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

Training participants: 12
Test participants: 3
Training rows: 65
Test rows: 9


In [35]:
from sklearn.ensemble import IsolationForest

basic_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

expanded_features = basic_features + [
    "steps_previous_3obs",
    "sleep_previous_3obs"
]

# Use the same test observations for both models
common_test = test_data.dropna(
    subset=expanded_features
).copy()

basic_train = train_data.dropna(
    subset=basic_features
)

expanded_train = train_data.dropna(
    subset=expanded_features
)

basic_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

expanded_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

basic_model.fit(basic_train[basic_features])
expanded_model.fit(expanded_train[expanded_features])

common_test["basic_label"] = basic_model.predict(
    common_test[basic_features]
)

common_test["expanded_label"] = expanded_model.predict(
    common_test[expanded_features]
)

print("Common test observations:", len(common_test))
print("\nBasic model:")
print(common_test["basic_label"].value_counts())

print("\nExpanded model:")
print(common_test["expanded_label"].value_counts())

print(
    "\nBoth flagged:",
    (
        (common_test["basic_label"] == -1)
        & (common_test["expanded_label"] == -1)
    ).sum()
)

Common test observations: 6

Basic model:
basic_label
-1    4
 1    2
Name: count, dtype: int64

Expanded model:
expanded_label
-1    4
 1    2
Name: count, dtype: int64

Both flagged: 3


In [36]:
comparison = pd.crosstab(
    common_test["basic_label"],
    common_test["expanded_label"],
    rownames=["Basic model"],
    colnames=["Expanded model"]
)

display(comparison)

print(
    "Total matching predictions:",
    (
        common_test["basic_label"]
        == common_test["expanded_label"]
    ).sum(),
    "out of",
    len(common_test)
)

Expanded model,-1,1
Basic model,,
-1,3,1
1,1,1


Total matching predictions: 4 out of 6


## Day 13 — Model Comparison

Both feature sets flagged 4 of the 6 common test observations as anomalies.
Three observations were flagged by both models.

Equal anomaly counts do not establish equivalent predictions or superior performance. The small test set and absence of ground-truth labels limit conclusions. The comparison is exploratory, not clinical validation.

In [37]:
from pathlib import Path
import joblib
from sklearn.ensemble import IsolationForest

basic_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

# Recreate the same participant-level training split
participant_ids = sorted(evaluation_data["Id"].unique())
test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

X_train = train_data[basic_features].dropna()

final_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

final_model.fit(X_train)

models_path = Path("../models")
models_path.mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "model": final_model,
        "features": basic_features
    },
    models_path / "isolation_forest.joblib"
)

print("Model saved successfully.")
print("Training rows:", len(X_train))
print("Features:", basic_features)

Model saved successfully.
Training rows: 65
Features: ['steps_deviation_7d_pct', 'sleep_deviation_7d_pct']


In [38]:

import sklearn
print(sklearn.__version__)

1.7.2


In [39]:
import sys
import sklearn

print("Python:", sys.executable)
print("scikit-learn:", sklearn.__version__)

Python: c:\Users\mukul\AppData\Local\Programs\Python\Python313\python.exe
scikit-learn: 1.7.2


In [40]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

print("Dataset shape:", feature_data.shape)
print("Unique participants:", feature_data["Id"].nunique())
print("Available columns:")
print(feature_data.columns.tolist())

Dataset shape: (199, 21)
Unique participants: 20
Available columns:
['Id', 'date', 'TotalSteps', 'Calories', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'recorded_sleep_minutes', 'baseline_steps_7d', 'baseline_sleep_7d', 'steps_deviation_7d', 'steps_deviation_7d_pct', 'sleep_deviation_7d', 'sleep_deviation_7d_pct', 'steps_rolling_3d', 'sleep_rolling_3d', 'activity_ratio', 'sleep_ratio', 'day_number', 'is_baseline']


In [41]:
import numpy as np
import pandas as pd

feature_columns = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

sequence_length = 3

sequence_data = feature_data.copy()
sequence_data["date"] = pd.to_datetime(sequence_data["date"])

sequence_data = sequence_data.sort_values(
    ["Id", "date"]
).reset_index(drop=True)

X_sequences = []
y_targets = []
participant_targets = []

for participant_id, person_data in sequence_data.groupby("Id"):
    person_data = person_data.sort_values("date")

    values = person_data[feature_columns].to_numpy()

    for i in range(sequence_length, len(values)):
        previous_observations = values[i-sequence_length:i]
        next_observation = values[i]

        if np.isnan(previous_observations).any():
            continue

        if np.isnan(next_observation).any():
            continue

        X_sequences.append(previous_observations)
        y_targets.append(next_observation)
        participant_targets.append(participant_id)

X_sequences = np.array(X_sequences, dtype=np.float32)
y_targets = np.array(y_targets, dtype=np.float32)

print("Input sequences shape:", X_sequences.shape)
print("Target shape:", y_targets.shape)
print("Unique target participants:", len(set(participant_targets)))

Input sequences shape: (140, 3, 2)
Target shape: (140, 2)
Unique target participants: 19


In [42]:
from sklearn.model_selection import GroupShuffleSplit

groups = np.array(participant_targets)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_indices, test_indices = next(
    splitter.split(X_sequences, y_targets, groups=groups)
)

X_train = X_sequences[train_indices]
X_test = X_sequences[test_indices]

y_train = y_targets[train_indices]
y_test = y_targets[test_indices]

train_participants = set(groups[train_indices])
test_participants = set(groups[test_indices])

print("Train sequences:", X_train.shape)
print("Test sequences:", X_test.shape)
print("Train participants:", len(train_participants))
print("Test participants:", len(test_participants))
print("Participant overlap:", train_participants & test_participants)

Train sequences: (103, 3, 2)
Test sequences: (37, 3, 2)
Train participants: 14
Test participants: 5
Participant overlap: set()


In [43]:
import torch
import torch.nn as nn

class HealthPatternModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 2, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


model = HealthPatternModel()

print(model)

HealthPatternModel(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=6, out_features=16, bias=True)
    (2): ReLU()
    (3): Linear(in_features=16, out_features=2, bias=True)
  )
)


In [44]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

feature_columns = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

sequence_data = feature_data.copy()
sequence_data["date"] = pd.to_datetime(sequence_data["date"])
sequence_data = sequence_data.sort_values(["Id", "date"])

X_sequences = []
y_targets = []
participant_targets = []

for participant_id, person_data in sequence_data.groupby("Id"):
    values = person_data[feature_columns].to_numpy()

    for i in range(3, len(values)):
        previous_observations = values[i-3:i]
        next_observation = values[i]

        if np.isnan(previous_observations).any():
            continue
        if np.isnan(next_observation).any():
            continue

        X_sequences.append(previous_observations)
        y_targets.append(next_observation)
        participant_targets.append(participant_id)

X_sequences = np.array(X_sequences, dtype=np.float32)
y_targets = np.array(y_targets, dtype=np.float32)
groups = np.array(participant_targets)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_indices, test_indices = next(
    splitter.split(X_sequences, y_targets, groups=groups)
)

X_train = X_sequences[train_indices]
X_test = X_sequences[test_indices]
y_train = y_targets[train_indices]
y_test = y_targets[test_indices]

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print(
    "Participant overlap:",
    set(groups[train_indices]) & set(groups[test_indices])
)

Train: (103, 3, 2) (103, 2)
Test: (37, 3, 2) (37, 2)
Participant overlap: set()


In [45]:
import torch
import torch.nn as nn

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Define loss and optimizer
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100

for epoch in range(epochs):
    model.train()

    predictions = model(X_train_tensor)
    loss = loss_function(predictions, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [20/100], Loss: 1267.2717
Epoch [40/100], Loss: 1195.6222
Epoch [60/100], Loss: 1141.5367
Epoch [80/100], Loss: 1099.7350
Epoch [100/100], Loss: 1067.6058


In [46]:
from sklearn.metrics import mean_absolute_error

model.eval()

with torch.no_grad():
    test_predictions = model(X_test_tensor)

test_predictions_np = test_predictions.numpy()
y_test_np = y_test_tensor.numpy()

mae = mean_absolute_error(
    y_test_np,
    test_predictions_np
)

steps_mae = mean_absolute_error(
    y_test_np[:, 0],
    test_predictions_np[:, 0]
)

sleep_mae = mean_absolute_error(
    y_test_np[:, 1],
    test_predictions_np[:, 1]
)

print(f"Overall MAE: {mae:.2f}")
print(f"Steps deviation MAE: {steps_mae:.2f} percentage points")
print(f"Sleep deviation MAE: {sleep_mae:.2f} percentage points")

Overall MAE: 21.84
Steps deviation MAE: 27.13 percentage points
Sleep deviation MAE: 16.54 percentage points


In [47]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# Predict the training-set mean for every test observation
mean_baseline_prediction = np.tile(
    y_train.mean(axis=0),
    (len(y_test), 1)
)

baseline_overall_mae = mean_absolute_error(
    y_test,
    mean_baseline_prediction
)

baseline_steps_mae = mean_absolute_error(
    y_test[:, 0],
    mean_baseline_prediction[:, 0]
)

baseline_sleep_mae = mean_absolute_error(
    y_test[:, 1],
    mean_baseline_prediction[:, 1]
)

print(f"Baseline overall MAE: {baseline_overall_mae:.2f}")
print(f"Baseline steps MAE: {baseline_steps_mae:.2f}")
print(f"Baseline sleep MAE: {baseline_sleep_mae:.2f}")

Baseline overall MAE: 22.50
Baseline steps MAE: 27.43
Baseline sleep MAE: 17.57


## Day 16 — Deep Learning Time-Series Experiment

### Objective
Predict the next observation's personal-baseline steps and recorded-sleep deviations using the previous three available observations.

### Dataset
- 199 participant-day observations across 20 participants.
- 140 generated sequences from 19 participants.
- Participant-level split: 103 training sequences and 37 test sequences.
- No participant overlap between training and testing.

### Model
A lightweight PyTorch feed-forward neural network:
- Input: 3 observations × 2 features
- Hidden layer: 16 units with ReLU activation
- Output: 2 predicted deviation values
- Training: 100 epochs using Adam and mean squared error loss.

### Evaluation
- Neural network overall MAE: 22.49
- Mean-baseline overall MAE: 22.50
- Steps MAE: 27.44 (network), 27.43 (baseline)
- Sleep MAE: 17.54 (network), 17.57 (baseline)

### Conclusion and Limitations
The neural network did not demonstrate a meaningful improvement over the simple mean baseline in this experiment. The dataset and test set are small, and the results are exploratory. Further experiments should consider feature scaling, stronger baselines, temporal models and repeated participant-level validation before drawing conclusions.

In [48]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("Train participants:", len(train_ids))
print("Test participants:", len(test_ids))
print("Participant overlap:", set(train_ids) & set(test_ids))

X_train: (103, 3, 2)
X_test: (37, 3, 2)
y_train: (103, 2)
y_test: (37, 2)


NameError: name 'train_ids' is not defined

In [ ]:
print("Train participants:", len(train_ids))
print("Test participants:", len(test_ids))
print("Participant overlap:", set(train_ids) & set(test_ids))

NameError: name 'train_ids' is not defined

In [49]:
print("target_participants exists:", "target_participants" in globals())
print("sequence_data exists:", "sequence_data" in globals())
print("GroupShuffleSplit exists:", "GroupShuffleSplit" in globals())

target_participants exists: False
sequence_data exists: True
GroupShuffleSplit exists: True


In [50]:
print("Columns:", sequence_data.columns.tolist())
print(sequence_data.head())

Columns: ['Id', 'date', 'TotalSteps', 'Calories', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'recorded_sleep_minutes', 'baseline_steps_7d', 'baseline_sleep_7d', 'steps_deviation_7d', 'steps_deviation_7d_pct', 'sleep_deviation_7d', 'sleep_deviation_7d_pct', 'steps_rolling_3d', 'sleep_rolling_3d', 'activity_ratio', 'sleep_ratio', 'day_number', 'is_baseline']
           Id       date  TotalSteps  Calories  VeryActiveMinutes  \
0  1503960366 2016-03-25       11004      1819                 33   
1  1503960366 2016-03-26       17609      2154                 89   
2  1503960366 2016-03-27       12736      1944                 56   
3  1503960366 2016-03-28       13231      1932                 39   
4  1503960366 2016-03-29       12041      1886                 28   

   FairlyActiveMinutes  LightlyActiveMinutes  SedentaryMinutes  \
0                   12                   205               804   
1                   17                   274     

In [51]:
GroupShuffleSplit

sklearn.model_selection._split.GroupShuffleSplit

In [52]:
train_indices, test_indices = next(
    splitter.split(X_sequences, y_targets, groups=target_participants)
)

NameError: name 'target_participants' is not defined

In [53]:
print("X_sequences:", X_sequences.shape)
print("y_targets:", y_targets.shape)
print("sequence_data rows:", len(sequence_data))
print("splitter exists:", "splitter" in globals())
print("train_indices exists:", "train_indices" in globals())
print("test_indices exists:", "test_indices" in globals())

X_sequences: (140, 3, 2)
y_targets: (140, 2)
sequence_data rows: 199
splitter exists: True
train_indices exists: True
test_indices exists: True


In [54]:
print("Train indices:", train_indices.shape)
print("Test indices:", test_indices.shape)

print("Train indices range:",
      train_indices.min(), "to", train_indices.max())

print("Test indices range:",
      test_indices.min(), "to", test_indices.max())

print("Overlap:",
      set(train_indices) & set(test_indices))

Train indices: (103,)
Test indices: (37,)
Train indices range: 12 to 139
Test indices range: 0 to 102
Overlap: set()


In [56]:
from sklearn.model_selection import GroupShuffleSplit

# Recover participant IDs for all 140 sequences
target_participants = []

for participant_id, participant_data in sequence_data.groupby("Id"):
    participant_data = participant_data.sort_values("date")

    if len(participant_data) >= 4:
        target_participants.extend(
            [participant_id] * (len(participant_data) - 3)
        )

print("Recovered sequence participants:", len(target_participants))

Recovered sequence participants: 140


In [57]:
train_participants = set(
    target_participants[i] for i in train_indices
)
test_participants = set(
    target_participants[i] for i in test_indices
)

print("Train participants:", len(train_participants))
print("Test participants:", len(test_participants))
print("Participant overlap:", train_participants & test_participants)

Train participants: 14
Test participants: 5
Participant overlap: set()


In [58]:
from sklearn.preprocessing import StandardScaler

# Sequence shape: (samples, timesteps, features)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

print("Original train shape:", X_train.shape)
print("Scaled train shape:", X_train_scaled.shape)
print("Original test shape:", X_test.shape)
print("Scaled test shape:", X_test_scaled.shape)

Original train shape: (103, 3, 2)
Scaled train shape: (103, 3, 2)
Original test shape: (37, 3, 2)
Scaled test shape: (37, 3, 2)


In [59]:
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
import numpy as np

# Convert scaled sequences into PyTorch tensors
X_train_scaled_tensor = torch.tensor(
    X_train_scaled, dtype=torch.float32
)
X_test_scaled_tensor = torch.tensor(
    X_test_scaled, dtype=torch.float32
)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create a fresh model
scaled_model = HealthPatternModel()

loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(
    scaled_model.parameters(), lr=0.001
)

epochs = 100

for epoch in range(epochs):
    scaled_model.train()

    predictions = scaled_model(X_train_scaled_tensor)
    loss = loss_function(predictions, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 20, Loss: 1202.7386
Epoch 40, Loss: 1200.5637
Epoch 60, Loss: 1198.1115
Epoch 80, Loss: 1195.3031
Epoch 100, Loss: 1192.0802


In [60]:
scaled_model.eval()

with torch.no_grad():
    scaled_predictions = scaled_model(X_test_scaled_tensor)

scaled_predictions_np = scaled_predictions.numpy()
y_test_np = y_test_tensor.numpy()

overall_mae_scaled = mean_absolute_error(
    y_test_np, scaled_predictions_np
)

steps_mae_scaled = mean_absolute_error(
    y_test_np[:, 0], scaled_predictions_np[:, 0]
)

sleep_mae_scaled = mean_absolute_error(
    y_test_np[:, 1], scaled_predictions_np[:, 1]
)

print(f"Scaled model overall MAE: {overall_mae_scaled:.4f}")
print(f"Steps deviation MAE: {steps_mae_scaled:.4f} percentage points")
print(f"Sleep deviation MAE: {sleep_mae_scaled:.4f} percentage points")

Scaled model overall MAE: 21.9334
Steps deviation MAE: 27.4363 percentage points
Sleep deviation MAE: 16.4306 percentage points


In [61]:
# Predict the training-set mean for every test sample
mean_baseline_prediction = np.tile(
    y_train.mean(axis=0),
    (len(y_test), 1)
)

baseline_overall_mae = mean_absolute_error(
    y_test_np, mean_baseline_prediction
)

baseline_steps_mae = mean_absolute_error(
    y_test_np[:, 0], mean_baseline_prediction[:, 0]
)

baseline_sleep_mae = mean_absolute_error(
    y_test_np[:, 1], mean_baseline_prediction[:, 1]
)

print(f"Mean baseline overall MAE: {baseline_overall_mae:.4f}")
print(f"Mean baseline steps MAE: {baseline_steps_mae:.4f}")
print(f"Mean baseline sleep MAE: {baseline_sleep_mae:.4f}")

print("\n--- Comparison ---")
print(f"Overall MAE improvement: {baseline_overall_mae - overall_mae_scaled:.4f}")
print(f"Steps MAE improvement: {baseline_steps_mae - steps_mae_scaled:.4f}")
print(f"Sleep MAE improvement: {baseline_sleep_mae - sleep_mae_scaled:.4f}")

Mean baseline overall MAE: 22.4989
Mean baseline steps MAE: 27.4298
Mean baseline sleep MAE: 17.5679

--- Comparison ---
Overall MAE improvement: 0.5654
Steps MAE improvement: -0.0064
Sleep MAE improvement: 1.1373


### Day 17: Feature Scaling and Evaluation

StandardScaler was fitted on the training sequences and applied to the test sequences using the same learned parameters. A fresh PyTorch neural network was trained on the scaled inputs.

The scaled model achieved an overall MAE of 21.9334, compared with 22.4989 for the training-mean baseline. Steps-deviation MAE remained nearly unchanged (27.4363 vs. 27.4298), while sleep-deviation MAE improved from 17.5679 to 16.4306.

These results provide preliminary evidence of improved sleep-deviation prediction, but not steps-deviation prediction. The small test set of 37 sequences from 5 participants limits the strength of the conclusion. Further evaluation is needed before claiming reliable generalization.